# 03 · Rentabilidad por ruta y buque

Compara contribución, EBITDA, margen y ocupación de equilibrio para decidir dónde actuar.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Rentabilidad por ruta

In [2]:
a=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
route=a.groupby(['route_id','route_name'],as_index=False).agg(revenue=('revenue','sum'),contribution=('contribution','sum'),ebitda=('ebitda','sum'),passengers=('passengers','sum'),voyages=('voyages','sum'),occupancy=('occupancy','mean'),break_even_occupancy=('break_even_occupancy','mean'))
route['ebitda_margin']=route.ebitda/route.revenue
route['safety_margin_pp']=(route.occupancy-route.break_even_occupancy)*100
route=route.sort_values('ebitda_margin',ascending=False)
route.to_csv(TABLES/'03_route_profitability.csv',index=False)
display(route)

  route_id          route_name  ...  ebitda_margin  safety_margin_pp
4  DEN-PMI      Dénia–Mallorca  ...           0.18              8.87
2  DEN-FOR    Dénia–Formentera  ...           0.08              4.47
5  VAL-IBZ      Valencia–Ibiza  ...           0.06              1.80
3  DEN-IBZ         Dénia–Ibiza  ...           0.06              2.69
1  BCN-PMI  Barcelona–Mallorca  ...           0.05              1.08
6  VAL-PMI   Valencia–Mallorca  ...           0.05              0.95
0  BCN-IBZ     Barcelona–Ibiza  ...           0.03              0.07

[7 rows x 11 columns]


## Matriz margen–volumen

In [3]:
plt.figure(figsize=(11,6))
sizes=route.revenue/25000
plt.scatter(route.occupancy*100,route.ebitda_margin*100,s=sizes,c=route.safety_margin_pp,cmap='RdYlGn',edgecolor='white',linewidth=1)
for _,r in route.iterrows(): plt.text(r.occupancy*100+.25,r.ebitda_margin*100,r.route_id,fontsize=9)
plt.axhline(12,color='#64748b',ls='--'); plt.xlabel('Ocupación media (%)'); plt.ylabel('Margen EBITDA (%)'); plt.title('Rentabilidad por ruta: ocupación, margen y escala')
plt.tight_layout(); plt.savefig(FIGURES/'03_matriz_rentabilidad_ruta.png',dpi=180,bbox_inches='tight'); plt.show()

## Rentabilidad por buque

In [4]:
vessel=a.groupby(['vessel_id','vessel_name'],as_index=False).agg(revenue=('revenue','sum'),ebitda=('ebitda','sum'),voyages=('voyages','sum'),fuel_tons=('fuel_tons','sum'))
vessel['ebitda_margin']=vessel.ebitda/vessel.revenue
vessel['fuel_tons_per_voyage']=vessel.fuel_tons/vessel.voyages
vessel.to_csv(TABLES/'03_vessel_profitability.csv',index=False)
display(vessel.sort_values('ebitda_margin',ascending=False))

  vessel_id     vessel_name  ...  ebitda_margin  fuel_tons_per_voyage
2       V03     Costa Dénia  ...           0.10                 35.00
3       V04    Mediterráneo  ...           0.10                 74.39
1       V02     Marina Azul  ...           0.06                 61.81
4       V05  Balear Express  ...           0.05                 67.30
0       V01    Levante Star  ...           0.04                 34.08

[5 rows x 8 columns]


## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.